# Notebook 1 — Export Marqo-FashionCLIP Vision Encoder to ONNX INT8

Exports only the **vision encoder** of Marqo-FashionCLIP.
Uses `open_clip` (the correct library for this model — not HuggingFace transformers).

**Output files (download at end):**
- `fashion_clip_vision.onnx` — full precision
- `fashion_clip_vision_int8.onnx` — INT8 quantized (~2-3x faster on CPU)

**Run on:** Colab T4 (free). GPU only speeds up model download — inference is CPU.

In [ ]:
!pip install -q open_clip_torch onnx onnxruntime Pillow numpy torch torchvision

In [ ]:
import torch
import numpy as np
from PIL import Image
import open_clip
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import time
import os

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "/content"  # Colab working directory — all files saved here

ONNX_PATH = f"{WORK_DIR}/fashion_clip_vision.onnx"
INT8_PATH = f"{WORK_DIR}/fashion_clip_vision_int8.onnx"

print(f"Device:   {DEVICE}")
print(f"Work dir: {WORK_DIR}")

In [ ]:
# ── Load Marqo-FashionCLIP via open_clip ──────────────────────────────────────
# Must use open_clip — this model is NOT compatible with HuggingFace transformers CLIPProcessor
print("Downloading Marqo-FashionCLIP (this may take a minute) ...")
model, preprocess = open_clip.create_model_from_pretrained('hf-hub:Marqo/marqo-fashionCLIP')
model = model.eval()

# The visual encoder is model.visual
vision_encoder = model.visual.eval()
print(f"Vision encoder params: {sum(p.numel() for p in vision_encoder.parameters()):,}")
print(f"Preprocess pipeline:\n{preprocess}")

In [ ]:
# ── Verify vision encoder output shape ───────────────────────────────────────
dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = vision_encoder(dummy)

# open_clip visual encoder returns a single tensor (B, embed_dim)
print(f"Output type:  {type(out)}")
print(f"Output shape: {out.shape}")
assert out.shape == (1, 512), f"Expected (1, 512), got {out.shape}"

In [ ]:
# ── Wrap for clean ONNX export ────────────────────────────────────────────────
class VisionWrapper(torch.nn.Module):
    """Thin wrapper so ONNX export has clean named inputs/outputs."""
    def __init__(self, visual):
        super().__init__()
        self.visual = visual

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.visual(pixel_values)  # (B, 512)

# Always export from CPU for maximum portability
wrapper = VisionWrapper(vision_encoder.cpu()).eval()

dummy_input = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    test_out = wrapper(dummy_input)
print(f"Wrapper output shape: {test_out.shape}")

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────────
print(f"Exporting to {ONNX_PATH} ...")

torch.onnx.export(
    wrapper,
    dummy_input,
    ONNX_PATH,
    input_names=["pixel_values"],
    output_names=["embedding"],
    dynamic_axes={
        "pixel_values": {0: "batch_size"},
        "embedding":    {0: "batch_size"},
    },
    opset_version=17,
    do_constant_folding=True,
)

assert os.path.isfile(ONNX_PATH), "Export failed — file not created"
onnx.checker.check_model(ONNX_PATH)
print(f"Saved and validated: {ONNX_PATH}")
print(f"Size: {os.path.getsize(ONNX_PATH) / 1e6:.1f} MB")

In [ ]:
# ── Validate: ONNX outputs must match PyTorch ─────────────────────────────────
sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

with torch.no_grad():
    pt_out = wrapper(torch.from_numpy(test_input)).numpy()

onnx_out = sess.run(["embedding"], {"pixel_values": test_input})[0]

max_diff = float(np.abs(pt_out - onnx_out).max())
print(f"Max difference PyTorch vs ONNX: {max_diff:.6f}")
assert max_diff < 1e-3, f"Outputs differ too much ({max_diff}) — check export"
print("Outputs match")

In [ ]:
# ── INT8 dynamic quantization ─────────────────────────────────────────────────
print(f"Quantizing to INT8 ...")
quantize_dynamic(ONNX_PATH, INT8_PATH, weight_type=QuantType.QInt8)

assert os.path.isfile(INT8_PATH), "Quantization failed — file not created"
print(f"Saved: {INT8_PATH}")
print(f"Size: {os.path.getsize(INT8_PATH) / 1e6:.1f} MB")

In [ ]:
# ── Benchmark latency on CPU ──────────────────────────────────────────────────
def benchmark(model_path, n_runs=50):
    sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    inp  = np.random.randn(1, 3, 224, 224).astype(np.float32)
    for _ in range(5):  # warmup
        sess.run(["embedding"], {"pixel_values": inp})
    t0 = time.perf_counter()
    for _ in range(n_runs):
        sess.run(["embedding"], {"pixel_values": inp})
    return (time.perf_counter() - t0) / n_runs * 1000

fp32_ms = benchmark(ONNX_PATH)
int8_ms = benchmark(INT8_PATH)

print(f"FP32 ONNX: {fp32_ms:.1f} ms/image")
print(f"INT8 ONNX: {int8_ms:.1f} ms/image")
print(f"Speedup:   {fp32_ms / int8_ms:.2f}x")

In [ ]:
# ── End-to-end test: real PIL image → embedding ───────────────────────────────
# Uses open_clip's own preprocess — guaranteed to match what the model expects

def embed_image(pil_img: Image.Image, sess: ort.InferenceSession) -> np.ndarray:
    """Preprocess a PIL image and return a 512-dim embedding."""
    tensor = preprocess(pil_img)          # torchvision transforms → (3, 224, 224)
    arr    = tensor.unsqueeze(0).numpy()  # (1, 3, 224, 224) float32
    return sess.run(["embedding"], {"pixel_values": arr})[0]  # (1, 512)

sess_int8 = ort.InferenceSession(INT8_PATH, providers=["CPUExecutionProvider"])

# Test with a solid colour patch
test_img = Image.new("RGB", (300, 400), color=(100, 150, 200))
emb = embed_image(test_img, sess_int8)

print(f"Embedding shape: {emb.shape}")
print(f"Embedding norm:  {np.linalg.norm(emb):.4f}")

# Print the preprocess parameters so embed_polyvore.ipynb can replicate them
print("\n--- Copy these preprocessing params into embed_polyvore.ipynb ---")
for t in preprocess.transforms:
    print(f"  {t}")

In [ ]:
# ── Download both model files ─────────────────────────────────────────────────
# Place both in backend/model/ in the app repo after downloading.
# fashion_clip_vision_int8.onnx → used in production (backend)
# fashion_clip_vision.onnx      → kept as reference / for retraining

from google.colab import files

print(f"Downloading {INT8_PATH} ...")
files.download(INT8_PATH)

print(f"Downloading {ONNX_PATH} ...")
files.download(ONNX_PATH)

print("Done. Next step: upload Cleaned-Maryland-Dataset to Kaggle and run embed_polyvore.ipynb")